# Pipeline Mapping Preconfiguration

AI SDK provides functionality for preconfiguring pipeline inputs, outputs, and topic-based parameters, preparing the required connection between the various devices and the pipeline through the AI Inference Server. In this tutorial, we show multiple examples how such a mapping preconfiguration can be done.

From this point on, we are referring to pipeline inputs and outputs collectively as pipeline variables, but this term does not include pipeline parameters.

## Mapping Pipeline Variables

### Connections and Tag Names

For the sake of this example, assume that our pipeline processes images through a IE Vision Connector, and also gets an additional temperature information from a PLC in payload format SIMATIC v1 through the Databus. The pipeline calculates an acceptance level value as a string, and sends it back to the Databus.

First, we have to define the required connections using `Connection` objects. A `Connection` represents a Data Connection defined in the AI Inference Server. The properties used for finding the proper Data Connection are Connection **Type** and **Payload** format (collectively determined by the `cptype` attribute of a `Connection` object). If they match on multiple connections, the name makes the difference.

In [ ]:
from simaticai.payloads import Connection, ConnectionTypeAndPayloadFormat as CPT

vision_connection = Connection(name="_IE Vision Connector", cptype=CPT.IE_Vision)
simatic_connection = Connection(name="_IE Databus", cptype=CPT.Databus_SIMATICv1)
string_connection = Connection(name="_IE Databus", cptype=CPT.Databus_String)

Our pipeline will contain a single Python component which processes our input data. We can already set the input variables.

In [ ]:
from simaticai.deployment import PythonComponent

component = PythonComponent(name="python_component")
# Setting various attributes of the component: resources, entrypoint, dependencies, etc.
# ...
component.add_input("vision_payload", _type="ImageSet")
component.add_input("temperature", _type="Double")
component.inputs

Since in our example this is the only component, we can add the output variable as well.

In [ ]:
component.add_output("acceptance", _type="String")
component.outputs

Let's create the pipeline from our single component.

In [ ]:
from simaticai.deployment import Pipeline

pipeline = Pipeline.from_components([component], name="test_pipeline")
pipeline

We can check the input and output variables of the pipeline by directly invoking the pipeline's lists.

In [ ]:
pipeline.inputs

In [ ]:
pipeline.outputs

As we can see, from the pipeline's perspective the inputs and outputs are `PipelineVariable` objects. These objects represent a variable in our pipeline, linking the component's inputs and outputs to a connection and optional mapping.

A `PipelineVariable` object has `componentName` and `variableName` attributes that help us identify the variable in our pipeline, and optional mapping attributes such as `connection`, `tagName`, `cameraName` and `mapping`. For IE Vision connections, `cameraName` should be defined, otherwise `tagName` is preferred. The `mapping` attribute describes the topic or camera id we want to map to.

If the connection and the camera name / tag name can identify the connection uniquely, we don't have to provide the mapping itself: AI Inference Server can determine this information.

In [ ]:
for _input in pipeline.inputs:
    name = _input.variableName
    if name == "vision_payload":
        _input.add_mapping_with_connection(vision_connection, cameraName="camera_1")
    if name == "temperature":
        _input.add_mapping_with_connection(simatic_connection, tagName="PLC_Temperature")

pipeline.inputs

Notice how in the `PipelineVariable` our camera name is actually represented in the `tagName` field.

Similarly we can configure our output variable as well.

In [ ]:
for _output in pipeline.outputs:
    name = _output.variableName
    if name == "acceptance":
        _output.add_mapping_with_connection(string_connection, tagName="acceptance_level_str")

pipeline.outputs

And we're done with the mapping preconfiguration!

However, it is possible that AI Inference Server cannot deduce the proper mapping from the connection and the tag name. In this case, we have to provide the mapping ourselves, as we'll see in the next example.

### User Defined Mappings

Assume we have two separate PLCs, each measuring some temperature value, and both uses the same Databus connection and payload types we established earlier. In this example we use the `mapping` attribute of the `PipelineVariable` objects. We have to know the topic and subtopic or id of these variables (id in case of SIMATIC v1 payloads).

Let's say we already collected the relevant information by acquiring the data points dictionary.

In [ ]:
data_points = {
    "dataPoints": [
        {
            "name": "PLC data points",
            "topic": "ie/d/j/simatic/v1/s7c1/dp/r/predictor",
            "dataPointDefinitions": [
                {"name": "temperature_from_plc_1", "id": "101", "dataType": "Double"},
                {"name": "temperature_from_plc_2", "id": "201", "dataType": "Double"},
            ],
        }
    ]
}

We can acquire the necessary data for our pipeline input variables.

In [ ]:
topic = data_points["dataPoints"][0]["topic"]

variable_1_name = data_points["dataPoints"][0]["dataPointDefinitions"][0]["name"]
variable_1_id = data_points["dataPoints"][0]["dataPointDefinitions"][0]["id"]
variable_1_type = data_points["dataPoints"][0]["dataPointDefinitions"][0]["dataType"]

variable_2_name = data_points["dataPoints"][0]["dataPointDefinitions"][1]["name"]
variable_2_id = data_points["dataPoints"][0]["dataPointDefinitions"][1]["id"]
variable_2_type = data_points["dataPoints"][0]["dataPointDefinitions"][1]["dataType"]

We are ready to configure our pipeline input mapping.

In [ ]:
component = PythonComponent(name="python_component")
# Setting various attributes of the component: resources, entrypoint, dependencies, etc.
# ...
component.add_input("temperature_from_plc_1", _type=variable_1_type)
component.add_input("temperature_from_plc_2", _type=variable_2_type)

pipeline = Pipeline.from_components([component], name="test_pipeline")

for _input in pipeline.inputs:
    if _input.variableName == variable_1_name:
        # mapping of temperature from PLC 1 is "ie/d/j/simatic/v1/s7c1/dp/r/predictor:101"
        _input.add_mapping_with_connection(simatic_connection, mapping=f"{topic}:{variable_1_id}")
    if _input.variableName == variable_2_name:
        # mapping of temperature from PLC 2 is "ie/d/j/simatic/v1/s7c1/dp/r/predictor:201"
        _input.add_mapping_with_connection(simatic_connection, mapping=f"{topic}:{variable_2_id}")

# setting pipeline output(s) similarly
# ...

pipeline.inputs

If `mapping` is set, `connection`, `tagName` and `cameraName` are all optional.

### Setting `PipelineVariables` Directly

Alternatively, we can set a series of `PipelineVariables` directly, and add them to the pipeline input and output lists.

Once again, assume that we collected the relevant information into dictionaries.

In [ ]:
input_variables = [
    {"name": "ph1", "type": "Double", "connection": "s7", "topic": "PLC1/r/pW1"},
    {"name": "ph2", "type": "Double", "connection": "s7", "topic": "PLC1/r/pW2"},
    {"name": "ph3", "type": "Double", "connection": "s7", "topic": "PLC1/r/pW3"},
]
output_variables = [
    {"name": "status", "type": "Integer", "connection": "str", "topic": "PLC1/w/Status"},
    {"name": "control", "type": "Integer", "connection": "s7", "topic": "PLC1/w/Control"},
]

We construct the component and the pipeline. This time we add a simple entrypoint ([entrypoint.py](./entrypoint.py)) to the component, so we will be able to see the generated pipeline configuration later.

In [ ]:
from simaticai.payloads.pipeline_variable import PipelineVariable
from pathlib import Path

component = PythonComponent(name="python_component")

component.add_resources(Path().resolve(), "entrypoint.py")
component.set_entrypoint("entrypoint.py")
# Setting various attributes of the component: resources, dependencies, etc.
# ...

pipeline = Pipeline(name="test_pipeline")
pipeline.add_component(component)

pipeline

We can set the different variables in both the component and the pipeline.

In [ ]:
variable_list = input_variables + output_variables
get_connection = lambda conn_str: simatic_connection if conn_str == "s7" else string_connection

for var in variable_list:
    pipeline_variable = PipelineVariable(
        componentName=component.name,
        variableName=var["name"],
        connection=get_connection(var["connection"]),
        mapping=f"{var["topic"]}:{var["name"]}"
    )

    if var in input_variables:
        component.add_input(var["name"], _type=var["type"])
        pipeline.inputs.append(pipeline_variable)
    if var in output_variables:
        component.add_output(var["name"], _type=var["type"])
        pipeline.outputs.append(pipeline_variable)

Let's see if they are set correctly.

In [ ]:
pipeline.inputs

In [ ]:
pipeline.outputs

Now that our pipeline is complete, we can check the pipeline config to see that our pipeline is properly prepared!

In [ ]:
from pprint import pprint  # pretty print

pipeline_config = pipeline.get_pipeline_config()
pprint(pipeline_config)

## Mapping Pipeline Parameters

It is possible that instead of a pipeline input, we need a parameter that we can use to tweak our system. A pipeline parameter can be either topic based, or non-topic based. Non-topic based parameters can be set on the AI Inference Server GUI, while topic based parameters arrive through a connection, similar to pipeline input and output variables. In either case, the pipeline parameter should have a default value, which our system can use initially.

We can use the pipeline's `add_parameter()` function, with the `topic_based` boolean parameter set accordingly. Let's say we have a `line_speed` parameter, representing the speed of the assembly line our pipeline monitors with a float number (`Double`), with a default value of `30.0`.

In [ ]:
pipeline.add_parameter(name="line_speed", default_value=30.0, type_name="Double", topic_based=True)
pipeline.parameters

We can see that our parameter is represented with a `PipelineParameter` object. `PipelineParameter` objects have `name` attribute to identify the parameter, and a default value. Optionally we can set the data type of the parameter (which `PipelineParameter` can deduce from the default value anyway), and a description for documentation purposes. If the parameter is topic based, we can add `connection`, `tagName` and `mapping`, similar to pipeline variables.

In [ ]:
parameter_connection = Connection(name="_IE Databus", cptype=CPT.Databus_SIMATICv1)
topic = "ie/d/j/simatic/v1/s7c1/dp/r/predictor"
line_speed_subtopic = "ls"

for param in pipeline.parameters:
    if param.name == "line_speed":
        param.add_mapping_with_connection(parameter_connection, mapping=f"{topic}/{line_speed_subtopic}")

pipeline.parameters

Alternatively, we can directly construct a `PipelineParameter` with every attribute set, and add it to the list of pipeline parameters. Let's use this form to add another parameter called `lighting_intensity`, representing the lighting conditions on our assembly line, that reads from the same topic.

In [ ]:
from simaticai.payloads.pipeline_variable import PipelineParameter

lighting_intensity_subtopic = "li"
param = PipelineParameter(name="light_intensity",
                          defaultValue=100.0,
                          dtype="Double",
                          topicBased=True,
                          connection=parameter_connection,
                          mapping=f"{topic}/{lighting_intensity_subtopic}")
pipeline.parameters.append(param)

pipeline.parameters

Our pipeline package is ready to export.

In [ ]:
edge_package_path = pipeline.export(destination='./packages')
edge_package_path